In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSV files
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Helper to compute summary for a dataframe given name column
def summarize(df, name_col=None):
    # total rows
    total_rows = len(df)
    # cmdb_id counts (top 20)
    if 'cmdb_id' in df.columns and total_rows > 0:
        cmdb_counts = df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count')
    else:
        cmdb_counts = pd.DataFrame(columns=['cmdb_id', 'count'])
    # name/metric counts (top 20) if requested
    if name_col and name_col in df.columns and total_rows > 0:
        name_counts = df[name_col].value_counts().rename_axis(name_col).reset_index(name='count')
    else:
        name_counts = pd.DataFrame(columns=[name_col or 'name', 'count'])
    # earliest and latest timestamps (convert epoch seconds to Asia/Shanghai timezone)
    if 'timestamp' in df.columns and total_rows > 0:
        # convert assuming timestamps are epoch seconds (UTC), then convert to Asia/Shanghai
        times = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        earliest = times.min()
        latest = times.max()
    else:
        earliest = pd.NaT
        latest = pd.NaT
    # compact summary DataFrame
    summary = pd.DataFrame([{
        'total_rows': total_rows,
        'earliest_ts': earliest,
        'latest_ts': latest
    }])
    return summary, cmdb_counts.head(20), name_counts.head(20)

# Summaries for each file
metric_summary, metric_cmdb_counts, metric_kpi_counts = summarize(metric_df, name_col='kpi_name')
trace_summary, trace_cmdb_counts, trace_name_counts = summarize(trace_df, name_col='trace_name')
log_summary, log_cmdb_counts, log_name_counts = summarize(log_df, name_col='log_name')
error_summary, error_cmdb_counts, error_name_counts = summarize(error_df, name_col='message')

# Display compact summaries (variables separated by commas as required)
metric_summary, metric_cmdb_counts, metric_kpi_counts, trace_summary, trace_cmdb_counts, trace_name_counts, log_summary, log_cmdb_counts, log_name_counts, error_summary, error_cmdb_counts

```
Out[1]:
```
```python
# Compose a plain-English compact summary using previously computed summary variables.
# The final output is a single string variable 'summary' for compact display.

def fmt_top(df, key_col='cmdb_id', top_n=5):
    if df is None or df.empty:
        return "None"
    rows = df.head(top_n)
    return ", ".join(f"{row[key_col]}:{int(row['count'])}" for _, row in rows.iterrows())

metric_earliest = metric_summary.loc[0, 'earliest_ts']
metric_latest = metric_summary.loc[0, 'latest_ts']
trace_earliest = trace_summary.loc[0, 'earliest_ts']
trace_latest = trace_summary.loc[0, 'latest_ts']
log_earliest = log_summary.loc[0, 'earliest_ts']
log_latest = log_summary.loc[0, 'latest_ts']
err_earliest = error_summary.loc[0, 'earliest_ts']
err_latest = error_summary.loc[0, 'latest_ts']

summary = (
    "Summary of telemetry files (compact):\n\n"
    "metric.csv:\n"
    f"- Total rows: {int(metric_summary.loc[0,'total_rows'])}\n"
    f"- Top cmdb_id (by rows, top 5): {fmt_top(metric_cmdb_counts, 'cmdb_id', 5)}\n"
    f"- Top kpi_name (top 5): {fmt_top(metric_kpi_counts, 'kpi_name', 5)}\n"
    f"- Time range: {metric_earliest} to {metric_latest}\n\n"
    "trace.csv:\n"
    f"- Total rows: {int(trace_summary.loc[0,'total_rows'])}\n"
    f"- Top cmdb_id (top 5): {fmt_top(trace_cmdb_counts, 'cmdb_id', 5)}\n"
    f"- Top trace_name (top 5): {fmt_top(trace_name_counts, 'trace_name', 5)}\n"
    f"- Time range: {trace_earliest} to {trace_latest}\n\n"
    "log.csv:\n"
    f"- Total rows: {int(log_summary.loc[0,'total_rows'])}\n"
    f"- Top cmdb_id (top 5): {fmt_top(log_cmdb_counts, 'cmdb_id', 5)}\n"
    f"- Top log_name (top 5): {fmt_top(log_name_counts, 'log_name', 5)}\n"
    f"- Time range: {log_earliest} to {log_latest}\n\n"
    "error_logs.csv:\n"
    f"- Total rows: {int(error_summary.loc[0,'total_rows'])}\n"
    f"- Top cmdb_id (top 5): {fmt_top(error_cmdb_counts, 'cmdb_id', 5)}\n"
    f"- (message counts not shown here; file used only for row counts/hosts)\n"
    f"- Time range: {err_earliest} to {err_latest}\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows               earliest_ts                 latest_ts
0       26552 2021-03-09 05:00:00+08:00 2021-03-09 05:30:00+08:00,          cmdb_id  count
0        Mysql02   4015
1        Mysql01   3903
2        Redis01   2250
3        Redis02   2198
4       Tomcat03   1700
5       Tomcat04   1588
6       Tomcat01   1517
7           IG02   1483
8           IG01   1461
9       apache02   1280
10      apache01   1252
11          MG01   1194
12          MG02   1191
13      Tomcat02    957
14  ServiceTest6     68
15      dockerA2     52
16  ServiceTest1     48
17  ServiceTest2     48
18  ServiceTest5     48
19  ServiceTest8     44,                                              kpi_name  count
0      OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    367
1       OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    362
2          OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    362
3                  OSLinux-OSLinux_ZABBIX_Host_Uptime    362
4        OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    361
5                          OSLinux-CPU_CPU_CPUSysTime    356
6                         OSLinux-CPU_CPU_CPUidleutil    356
7                         OSLinux-CPU_CPU_CPUUserTime    356
8                             OSLinux-CPU_CPU_CPULoad    356
9                              OSLinux-CPU_CPU_CPUWio    356
10  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...    354
11       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc    302
12                         OSLinux-CPU_CPU_CPUCpuUtil    292
13       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc    289
14              OSLinux-OSLinux_MEMORY_MEMORY_UserMem    273
15          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem    271
16           OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem    270
17             OSLinux-OSLinux_MEMORY_MEMORY_CacheMem    270
18  OSLinux-OSLinux_FILE_-tmp-zabbix_agentd.log_Fi...    255
19    OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum    233,    total_rows               earliest_ts                 latest_ts
0        3270 2021-03-09 05:00:00+08:00 2021-03-09 05:29:00+08:00,      cmdb_id  count
0       MG02    528
1       MG01    519
2   Tomcat04    330
3   Tomcat03    291
4   Tomcat01    288
5       IG01    261
6   dockerB2    252
7       IG02    246
8   dockerB1    234
9   dockerA2    141
10  dockerA1    123
11  Tomcat02     57,                        trace_name  count
0        trace.self.duration_mean    263
1         trace.self.duration_p95    263
2            trace.self.row_count    263
3         trace.to_MG02.row_count     95
4      trace.to_MG02.duration_p95     95
5     trace.to_MG02.duration_mean     95
6      trace.to_MG01.duration_p95     89
7         trace.to_MG01.row_count     89
8     trace.to_MG01.duration_mean     89
9   trace.from_IG01.duration_mean     59
10      trace.from_IG01.row_count     59
11   trace.from_IG01.duration_p95     59
12  trace.from_MG01.duration_mean     57
13      trace.from_MG01.row_count     57
14   trace.from_MG01.duration_p95     57
15  trace.from_MG02.duration_mean     54
16  trace.from_IG02.duration_mean     54
17      trace.from_IG02.row_count     54
18      trace.from_MG02.row_count     54
19   trace.from_IG02.duration_p95     54,    total_rows               earliest_ts                 latest_ts
0         346 2021-03-09 05:00:00+08:00 2021-03-09 05:29:00+08:00,     cmdb_id  count
0  Tomcat02     90
1  Tomcat04     56
2  apache02     54
3  apache01     52
4  Tomcat01     47
5  Tomcat03     47,            log_name  count
0   log.error_count    156
1     log.row_count    156
2  log.gc_pause_max     34,    total_rows               earliest_ts                 latest_ts
0          42 2021-03-09 05:00:01+08:00 2021-03-09 05:29:56+08:00,     cmdb_id  count
0  Tomcat02     34
1  apache02      4
2  apache01      4)```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse metric_df if available in kernel; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# Ensure correct dtypes
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')

# 1) Compute global P95 per (cmdb_id, kpi_name) using ALL rows (global thresholds before windowing)
p95_df = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'p95'})
)

# 2) Prepare datetime column in Asia/Shanghai timezone and filter to incident window (inclusive)
metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

start = pd.Timestamp('2021-03-09 05:00:00').tz_localize(tz)
end = pd.Timestamp('2021-03-09 05:30:00').tz_localize(tz)

window_df = metric_df[(metric_df['dt'] >= start) & (metric_df['dt'] <= end)].copy()

# 3) Merge windowed rows with global p95 for convenient per-row exceed checks
window_with_p95 = window_df.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# boolean exceed column
window_with_p95['exceed'] = window_with_p95['value'] >= window_with_p95['p95']

# 4) Aggregate per (cmdb_id, kpi_name) within the window
window_agg = (
    window_with_p95
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .agg(
        total_points_in_window=('value', 'size'),
        count_exceed=('exceed', 'sum'),
        earliest_exceed_ts=('dt', lambda s: s[window_with_p95.loc[s.index, 'exceed']].min() if s.size>0 else pd.NaT),
        max_value_in_window=('value', 'max')
    )
    .reset_index()
)

# 5) Left-join global p95s with window aggregates so that pairs with zero window points are included
full_agg = p95_df.merge(window_agg, on=['cmdb_id', 'kpi_name'], how='left')

# Fill missing window stats for pairs with no points in the window
full_agg['total_points_in_window'] = full_agg['total_points_in_window'].fillna(0).astype(int)
full_agg['count_exceed'] = full_agg['count_exceed'].fillna(0).astype(int)
# earliest_exceed_ts keep as timezone-aware datetime or NaT
# max_value_in_window may be NaN for groups with no points

# 6) Sort by count_exceed desc, then by max_value_in_window desc; show top 20 rows only
result_top = (
    full_agg
    .sort_values(by=['count_exceed', 'max_value_in_window'], ascending=[False, False])
    .loc[:, ['cmdb_id', 'kpi_name', 'p95', 'total_points_in_window', 'count_exceed', 'earliest_exceed_ts', 'max_value_in_window']]
    .head(20)
)

# Keep the full intermediate table cached as 'full_agg' per instructions (do not print it)
full_table = full_agg  # cached full intermediate table

# Display only the compact requested table (result_top)
result_top

```
Out[2]:
```
conclusion = (
    "Using metric.csv P95 thresholds computed from the entire series and filtering to the incident window "
    "2021-03-09 05:00:00+08:00 through 2021-03-09 05:30:00+08:00: The top 20 (cmdb_id, kpi_name) pairs all have 31 points in the window and "
    "count_exceed = 31, meaning every sample in the 31-minute window met or exceeded the global P95. "
    "The earliest exceedance for these top entries is at 2021-03-09 05:00:00+08:00. "
    "Affected components include IG01/IG02 and MG01/MG02 (JVM Memory metrics like HeapMemoryMax and NoHeapMemoryUsed), "
    "Tomcat nodes (Tomcat01/03/04) with high request MaxTime, ErrorCount, and Thread metrics, plus Mysql02 and apache02 process counts. "
    "These metrics were consistently at or above their historical P95 during the entire incident window, indicating persistent high values for those KPIs."
)
conclusion

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name           p95  total_points_in_window  count_exceed        earliest_exceed_ts  max_value_in_window
0         IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09                      31            31 2021-03-09 05:00:00+08:00         2.040136e+09
88        IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09                      31            31 2021-03-09 05:00:00+08:00         2.040136e+09
176       MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09                      31            31 2021-03-09 05:00:00+08:00         2.040136e+09
270       MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09                      31            31 2021-03-09 05:00:00+08:00         2.040136e+09
179       MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.069049e+08                      31            31 2021-03-09 05:00:00+08:00         5.069049e+08
3         IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.786755e+07                      31            31 2021-03-09 05:00:00+08:00         5.786755e+07
1171  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  4.053960e+05                      31            31 2021-03-09 05:00:00+08:00         4.053960e+05
1366  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  3.998460e+05                      31            31 2021-03-09 05:00:00+08:00         3.998460e+05
1475  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  3.685540e+05                      31            31 2021-03-09 05:00:00+08:00         3.685540e+05
1170  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  6.420000e+02                      31            31 2021-03-09 05:00:00+08:00         6.420000e+02
1474  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  6.400000e+02                      31            31 2021-03-09 05:00:00+08:00         6.400000e+02
1365  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  5.580000e+02                      31            31 2021-03-09 05:00:00+08:00         5.580000e+02
1185  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02                      31            31 2021-03-09 05:00:00+08:00         5.000000e+02
1383  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02                      31            31 2021-03-09 05:00:00+08:00         5.000000e+02
1492  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02                      31            31 2021-03-09 05:00:00+08:00         5.000000e+02
1183  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.000000e+02                      31            31 2021-03-09 05:00:00+08:00         1.000000e+02
1381  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.000000e+02                      31            31 2021-03-09 05:00:00+08:00         1.000000e+02
1490  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.000000e+02                      31            31 2021-03-09 05:00:00+08:00         1.000000e+02
804    Mysql02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  2.400000e+01                      31            31 2021-03-09 05:00:00+08:00         2.400000e+01
1644  apache02   OSLinux-OSLinux_PROCESS_apache_10001_PROCPPCount  6.000000e+00                      31            31 2021-03-09 05:00:00+08:00         6.000000e+00```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# reuse trace_df if available; otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

tz = pytz.timezone('Asia/Shanghai')

# Ensure correct dtypes
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')

# 1) Compute global P95 per (cmdb_id, trace_name) using ALL rows (global thresholds BEFORE windowing)
trace_p95 = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'p95'})
)

# 2) Convert timestamps to Asia/Shanghai timezone-aware datetimes and filter to incident window (inclusive)
trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
start = pd.Timestamp('2021-03-09 05:00:00').tz_localize(tz)
end = pd.Timestamp('2021-03-09 05:30:00').tz_localize(tz)
trace_window = trace_df[(trace_df['dt'] >= start) & (trace_df['dt'] <= end)].copy()

# 3) Merge windowed rows with global p95 for per-row exceed checks
trace_window = trace_window.merge(trace_p95, on=['cmdb_id', 'trace_name'], how='left')
trace_window['exceed'] = trace_window['value'] >= trace_window['p95']

# 4) Aggregate per (cmdb_id, trace_name) within the window
def earliest_exceed(series_dt, series_exceed):
    if series_exceed.any():
        return series_dt[series_exceed].min()
    else:
        return pd.NaT

agg_window = (
    trace_window
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .agg(
        total_points_in_window=('value', 'size'),
        count_exceed=('exceed', 'sum'),
        earliest_exceed_ts=('dt', lambda s: earliest_exceed(s, trace_window.loc[s.index, 'exceed'])),
        max_value_in_window=('value', 'max')
    )
    .reset_index()
)

# 5) Left-join global p95s with window aggregates so pairs with zero window points are included
full_trace_agg = trace_p95.merge(agg_window, on=['cmdb_id', 'trace_name'], how='left')

# Fill missing window stats for pairs with no points in the window
full_trace_agg['total_points_in_window'] = full_trace_agg['total_points_in_window'].fillna(0).astype(int)
full_trace_agg['count_exceed'] = full_trace_agg['count_exceed'].fillna(0).astype(int)
# earliest_exceed_ts remains NaT if missing; max_value_in_window may be NaN for no points

# Cache the full intermediate table as 'full_trace_agg' per instructions (do not print it)
full_trace_agg_cached = full_trace_agg

# 6) Sort by count_exceed desc, then max_value_in_window desc; show top 20 rows only and keep output compact
result_top_trace = (
    full_trace_agg_cached
    .sort_values(by=['count_exceed', 'max_value_in_window'], ascending=[False, False])
    .loc[:, ['cmdb_id', 'trace_name', 'p95', 'total_points_in_window', 'count_exceed', 'earliest_exceed_ts', 'max_value_in_window']]
    .head(20)
)

result_top_trace

```
Out[3]:
```
summary = (
    "Analysis using global P95 (computed from entire trace.csv) and the incident window 2021-03-09 05:00–05:30+08:\n"
    "- Top findings: several trace (cmdb_id, trace_name) pairs exceeded their global P95 multiple times in the window.\n"
    "- Highest repeated exceedances (count_exceed = 3) occur for row_count traces between IG and Tomcat/docker: "
    "IG01 -> trace.to_Tomcat01.row_count (20 pts, 3 exceeds, earliest 05:06, max=6), Tomcat01 <- trace.from_IG01.row_count (20 pts, 3, earliest 05:06, max=6), "
    "IG01 -> trace.to_Tomcat04.row_count and Tomcat04 <- trace.from_IG01.row_count (22 pts each, 3 exceeds, earliest 05:02, max=5), "
    "MG02 -> trace.to_dockerB2.row_count and dockerB2 <- trace.from_MG02.row_count (17 pts each, 3 exceeds, earliest 05:02, max=3), "
    "and two short-series pairs MG01/dockerA2 (3 pts each, 3 exceeds, earliest 05:06, max=1).\n"
    "- Next group (count_exceed = 2) includes several self.row_count and duration metrics on dockerB1/dockerB2/Tomcat nodes and IG nodes (earliest exceed times mostly between 05:02 and 05:27, with max values up to ~185 for dockerB1 self.row_count and ~82 for Tomcat01 self.row_count).\n"
    "- Interpretation: the window shows transient but repeated spikes in request counts and some elevated durations affecting IG, Tomcat, MG and dockerB services, with first exceedances generally around 05:02–05:06. These traces are the top candidates for investigation."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                     trace_name         p95  total_points_in_window  count_exceed        earliest_exceed_ts  max_value_in_window
5        IG01    trace.to_Tomcat01.row_count    6.000000                      20             3 2021-03-09 05:06:00+08:00              6.00000
98   Tomcat01      trace.from_IG01.row_count    6.000000                      20             3 2021-03-09 05:06:00+08:00              6.00000
11       IG01    trace.to_Tomcat04.row_count    4.000000                      22             3 2021-03-09 05:02:00+08:00              5.00000
134  Tomcat04      trace.from_IG01.row_count    4.000000                      22             3 2021-03-09 05:02:00+08:00              5.00000
95       MG02    trace.to_dockerB2.row_count    3.000000                      17             3 2021-03-09 05:02:00+08:00              3.00000
197  dockerB2      trace.from_MG02.row_count    3.000000                      17             3 2021-03-09 05:02:00+08:00              3.00000
38       MG01  trace.from_dockerA2.row_count    1.000000                       3             3 2021-03-09 05:06:00+08:00              1.00000
173  dockerA2        trace.to_MG01.row_count    1.000000                       3             3 2021-03-09 05:06:00+08:00              1.00000
185  dockerB1           trace.self.row_count  136.000000                      21             2 2021-03-09 05:02:00+08:00            185.00000
200  dockerB2           trace.self.row_count   75.850000                      22             2 2021-03-09 05:02:00+08:00            126.00000
104  Tomcat01           trace.self.row_count   57.800000                      23             2 2021-03-09 05:06:00+08:00             82.00000
125  Tomcat03           trace.self.row_count   63.600000                      23             2 2021-03-09 05:06:00+08:00             69.00000
1        IG01        trace.self.duration_p95   65.224815                      28             2 2021-03-09 05:03:00+08:00             67.12115
13       IG02        trace.self.duration_p95   60.000880                      28             2 2021-03-09 05:26:00+08:00             67.09100
140  Tomcat04           trace.self.row_count   54.000000                      26             2 2021-03-09 05:06:00+08:00             61.00000
12       IG02       trace.self.duration_mean   18.473534                      28             2 2021-03-09 05:26:00+08:00             60.00050
0        IG01       trace.self.duration_mean   35.132450                      28             2 2021-03-09 05:27:00+08:00             60.00000
2        IG01           trace.self.row_count   31.900000                      28             2 2021-03-09 05:06:00+08:00             42.00000
14       IG02           trace.self.row_count   34.600000                      28             2 2021-03-09 05:02:00+08:00             42.00000
83       MG02           trace.self.row_count   27.800000                      27             2 2021-03-09 05:02:00+08:00             34.00000```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# reuse existing variables if present
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure types
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')
log_df['timestamp'] = pd.to_numeric(log_df['timestamp'], errors='coerce').astype('Int64')

# 1) Compute global P95 per (cmdb_id, log_name) using ALL rows (global threshold BEFORE windowing)
log_p95 = (
    log_df
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'p95'})
)

# 2) Convert timestamps to Asia/Shanghai timezone-aware datetimes and filter to incident window (inclusive)
log_df['dt'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
start = pd.Timestamp('2021-03-09 05:00:00').tz_localize(tz)
end = pd.Timestamp('2021-03-09 05:30:00').tz_localize(tz)
log_window = log_df[(log_df['dt'] >= start) & (log_df['dt'] <= end)].copy()

# 3) Merge windowed rows with global p95 and compute exceed flag
log_window = log_window.merge(log_p95, on=['cmdb_id', 'log_name'], how='left')
log_window['exceed'] = log_window['value'] >= log_window['p95']

# 4) Aggregate per (cmdb_id, log_name) within the window
def earliest_exceed(series_dt):
    # series_dt is the dt Series for the group; use its index to reference the exceed mask in log_window
    idx = series_dt.index
    mask = log_window.loc[idx, 'exceed']
    if mask.any():
        return series_dt[mask].min()
    else:
        return pd.NaT

agg_window = (
    log_window
    .groupby(['cmdb_id', 'log_name'], dropna=False)
    .agg(
        total_points_in_window=('value', 'size'),
        count_exceed=('exceed', 'sum'),
        earliest_exceed_ts=('dt', earliest_exceed),
        max_value_in_window=('value', 'max')
    )
    .reset_index()
)

# 5) Left-join global p95s with window aggregates so pairs with zero window points are included
full_log_agg = log_p95.merge(agg_window, on=['cmdb_id', 'log_name'], how='left')

# Fill missing window stats for pairs with no points in the window
full_log_agg['total_points_in_window'] = full_log_agg['total_points_in_window'].fillna(0).astype(int)
full_log_agg['count_exceed'] = full_log_agg['count_exceed'].fillna(0).astype(int)
# earliest_exceed_ts stays NaT if missing; max_value_in_window may be NaN

# Cache full intermediate table as required (variable: full_log_agg)

# 6) Sort by count_exceed desc then max_value_in_window desc and return top 20 rows only (compact)
result_top_log = (
    full_log_agg
    .sort_values(by=['count_exceed', 'max_value_in_window'], ascending=[False, False])
    .loc[:, ['cmdb_id', 'log_name', 'p95', 'total_points_in_window', 'count_exceed', 'earliest_exceed_ts', 'max_value_in_window']]
    .head(20)
)

result_top_log

```
Out[4]:
```
Summary of log.csv analysis (global P95 across entire file, incident window 2021-03-09 05:00–05:30 +08):

- Method: global P95 was computed per (cmdb_id, log_name) from the entire log.csv series, then the file was filtered to the 05:00–05:30 window and per-pair statistics were computed (points in window, how many points >= global P95, first exceed timestamp, max in window).

Key compact findings (top candidates by count_exceed then max value):
- Tomcat04 / log.error_count: p95 = 0.0, 27 points in window, count_exceed = 27, earliest exceed 2021-03-09 05:01+08, max = 0.00.
- Tomcat01 / log.error_count: p95 = 0.0, 23 pts, count_exceed = 23, earliest 05:00+08, max = 0.00.
- Tomcat03 / log.error_count: p95 = 0.0, 23 pts, count_exceed = 23, earliest 05:01+08, max = 0.00.
- Tomcat04 / log.row_count: p95 = 8.0, 27 pts, count_exceed = 5, earliest 05:02+08, max = 9.00.
- apache01 / log.error_count: p95 = 1.0, 26 pts, count_exceed = 4, earliest 05:02+08, max = 1.00.
- apache02 / log.error_count: p95 = 1.0, 27 pts, count_exceed = 3, earliest 05:01+08, max = 2.00.
- Tomcat02 / log.row_count: p95 = 47.2, 30 pts, count_exceed = 2, earliest 05:28+08, max = 72.00.
- apache01 / log.row_count: p95 = 18.75, 26 pts, count_exceed = 2, earliest 05:02+08, max = 26.00.
- Tomcat02 / log.error_count: p95 = 3.55, 30 pts, count_exceed = 2, earliest 05:25+08, max = 18.00.
- Several Tomcat nodes report occasional gc_pause_max exceedances (small numeric values), each with 1–2 exceed events in the window.

Interpretation / notes:
- Several entries (Tomcat error_count) have global p95 = 0.0 and show count_exceed equal to the window point count because values in the window are 0 — this is a byproduct of the p95 baseline being zero, so these "exceeds" do not necessarily indicate new errors (they reflect that p95 is zero and observed values are >= 0).
- More actionable spikes: log.row_count and some error_count series for apache/Tomcat/Tomcat02 show concrete increases (e.g., Tomcat02 row_count max 72 at 05:28, Tomcat04 row_count max 9 at 05:02) that merit investigation.
- Earliest exceed times for notable spikes are mostly between ~05:01 and 05:28+08; focus investigation on Tomcat nodes and apache row_count/error patterns around those times.

If you want, I can list the full top-20 table with exact numeric columns or expand investigation to correlate these log spikes with metric/trace anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id          log_name      p95  total_points_in_window  count_exceed        earliest_exceed_ts  max_value_in_window
9   Tomcat04   log.error_count   0.0000                      27            27 2021-03-09 05:01:00+08:00                 0.00
0   Tomcat01   log.error_count   0.0000                      23            23 2021-03-09 05:00:00+08:00                 0.00
6   Tomcat03   log.error_count   0.0000                      23            23 2021-03-09 05:01:00+08:00                 0.00
11  Tomcat04     log.row_count   8.0000                      27             5 2021-03-09 05:02:00+08:00                 9.00
12  apache01   log.error_count   1.0000                      26             4 2021-03-09 05:02:00+08:00                 1.00
14  apache02   log.error_count   1.0000                      27             3 2021-03-09 05:01:00+08:00                 2.00
5   Tomcat02     log.row_count  47.2000                      30             2 2021-03-09 05:28:00+08:00                72.00
13  apache01     log.row_count  18.7500                      26             2 2021-03-09 05:02:00+08:00                26.00
3   Tomcat02   log.error_count   3.5500                      30             2 2021-03-09 05:25:00+08:00                18.00
15  apache02     log.row_count  14.8000                      27             2 2021-03-09 05:06:00+08:00                18.00
2   Tomcat01     log.row_count   7.8000                      23             2 2021-03-09 05:06:00+08:00                13.00
8   Tomcat03     log.row_count   9.9000                      23             2 2021-03-09 05:02:00+08:00                10.00
4   Tomcat02  log.gc_pause_max   2.1455                      30             2 2021-03-09 05:02:00+08:00                 2.32
10  Tomcat04  log.gc_pause_max   0.0790                       2             1 2021-03-09 05:22:00+08:00                 0.08
1   Tomcat01  log.gc_pause_max   0.0700                       1             1 2021-03-09 05:08:00+08:00                 0.07
7   Tomcat03  log.gc_pause_max   0.0600                       1             1 2021-03-09 05:11:00+08:00                 0.06```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
import re

# Reuse existing variables if present
try:
    error_df
except NameError:
    error_df = pd.read_csv('error_logs.csv')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure proper dtypes and timezone-aware datetime (UTC -> convert to Asia/Shanghai)
error_df['timestamp'] = pd.to_numeric(error_df['timestamp'], errors='coerce').astype('Int64')
error_df['dt'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Define incident window (inclusive)
start = pd.Timestamp('2021-03-09 05:00:00').tz_localize(tz)
end = pd.Timestamp('2021-03-09 05:30:00').tz_localize(tz)

# Filter to window and cache
error_window_df = error_df[(error_df['dt'] >= start) & (error_df['dt'] <= end)].copy()

# 1) total rows in-window
total_in_window = len(error_window_df)

# 2) top cmdb_id by row count (top 20)
top_cmdb_counts = (
    error_window_df['cmdb_id']
    .value_counts()
    .rename_axis('cmdb_id')
    .reset_index(name='count')
    .head(20)
)

# 3) for each of top 5 cmdb_id, earliest and latest timestamp in-window
top5_cmdb = top_cmdb_counts['cmdb_id'].head(5).tolist()
top5_times = (
    error_window_df[error_window_df['cmdb_id'].isin(top5_cmdb)]
    .groupby('cmdb_id', dropna=False)['dt']
    .agg(earliest_in_window='min', latest_in_window='max')
    .reset_index()
    .merge(top_cmdb_counts[['cmdb_id','count']], on='cmdb_id', how='left')
    .sort_values('count', ascending=False)
)

# 4) top 20 most frequent message texts (truncate each message to first 200 chars) with counts
# Use full message for counting, but display truncated form
error_window_df['message'] = error_window_df['message'].astype(str)
msg_counts = (
    error_window_df
    .groupby('message', dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)
msg_counts['message_trunc_200'] = msg_counts['message'].str.slice(0, 200)

top_messages = msg_counts[['message_trunc_200', 'count']].rename(columns={'message_trunc_200':'message_first200'})

# Keyword search (case-insensitive) within in-window messages
keywords = [
    r'OutOfMemory',
    r'OOM',
    r'OutOfMemoryError',
    r'timeout',
    r'timed out',
    r'connection refused',
    r'ConnectionReset',
    r'Exception',
    r'ERROR',
    r'java\.lang'
]
pattern = r'(' + r'|'.join(keywords) + r')'

keyword_mask = error_window_df['message'].str.contains(pattern, case=False, regex=True, na=False)
keyword_matches = error_window_df.loc[keyword_mask, ['dt', 'cmdb_id', 'message']].copy()
# Truncate message to first 500 chars for display if very long
keyword_matches['message_display'] = keyword_matches['message'].str.slice(0, 500)
keyword_matches = keyword_matches.sort_values('dt').head(20)

# Cache intermediate results (already in variables): error_window_df, msg_counts, keyword_matches

# Compact outputs: total count (scalar), top cmdb counts, top5 times, top 20 messages (truncated), and up to 20 keyword match rows
total_in_window, top_cmdb_counts, top5_times, top_messages, keyword_matches[['dt','cmdb_id','message_display']]

```
Out[5]:
```
```python
# Plain-English compact summary of error_logs.csv in the incident window
summary = (
    "Error logs in-window (2021-03-09 05:00:00+08 through 2021-03-09 05:30:00+08):\n\n"
    "1) Total rows in-window: 42\n\n"
    "2) Top hosts by row count (top 20 shown; only 3 present):\n"
    "   - Tomcat02: 34\n"
    "   - apache02: 4\n"
    "   - apache01: 4\n\n"
    "3) Earliest and latest timestamps in-window for top 5 hosts (only 3 present):\n"
    "   - Tomcat02: earliest 2021-03-09 05:00:01+08:00, latest 2021-03-09 05:29:56+08:00 (34 rows)\n"
    "   - apache01: earliest 2021-03-09 05:02:12+08:00, latest 2021-03-09 05:29:03+08:00 (4 rows)\n"
    "   - apache02: earliest 2021-03-09 05:01:49+08:00, latest 2021-03-09 05:29:04+08:00 (4 rows)\n\n"
    "4) Top 20 most frequent messages (truncated to first 200 chars) with counts:\n"
    "   - Most messages are unique (count = 1). Examples include many IG01 POST requests such as\n"
    "     \"IG01 POST /UOCP/base/ServiceTest2.json HTTP/1...\" and similar POSTs to ServiceTest and person endpoints\n"
    "     (each listed message has count 1 in the top-20 sample).\n\n"
    "Keyword search (case-insensitive) within in-window messages for terms: OutOfMemory, OOM, OutOfMemoryError, timeout, "
    "timed out, connection refused, ConnectionReset, Exception, ERROR, java.lang:\n"
    "   - No matches found in the filtered window (0 matching rows).\n\n"
    "Interpretation / next steps:\n"
    "   - Tomcat02 is the dominant source of error-log entries during the window (34 of 42 rows). Investigate Tomcat02 logs around the timestamps shown.\n"
    "   - The messages in the top list are largely unique POST requests (ServiceTest/person endpoints) rather than repeated stack traces or OOM/timeout keywords.\n"
    "   - Since no OOM/timeout/Exception keywords were found in-window, focus on request-level issues or upstream traffic patterns affecting Tomcat02 and the apache nodes.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(42,     cmdb_id  count
0  Tomcat02     34
1  apache02      4
2  apache01      4,     cmdb_id        earliest_in_window          latest_in_window  count
0  Tomcat02 2021-03-09 05:00:01+08:00 2021-03-09 05:29:56+08:00     34
1  apache01 2021-03-09 05:02:12+08:00 2021-03-09 05:29:03+08:00      4
2  apache02 2021-03-09 05:01:49+08:00 2021-03-09 05:29:04+08:00      4,                                      message_first200  count
0   IG01 POST /UOCP/base/ServiceTest2.json HTTP/1....      1
1   IG01 POST /UOCP/base/ServiceTest4.json HTTP/1....      1
2   IG01 POST /UOCP/base/ServiceTest4.json HTTP/1....      1
3   IG01 POST /UOCP/base/ServiceTest4.json HTTP/1....      1
4   IG01 POST /UOCP/base/ServiceTest6.json HTTP/1....      1
5   IG01 POST /UOCP/base/ServiceTest6.json HTTP/1....      1
6   IG01 POST /UOCP/base/ServiceTest8.json HTTP/1....      1
7   IG01 POST /UOCP/base/ServiceTest8.json HTTP/1....      1
8   IG01 POST /UOCP/person/ServiceTest1.json HTTP/...      1
9   IG01 POST /UOCP/person/ServiceTest1.json HTTP/...      1
10  IG01 POST /UOCP/person/ServiceTest10.json HTTP...      1
11  IG01 POST /UOCP/person/ServiceTest3.json HTTP/...      1
12  IG01 POST /UOCP/person/ServiceTest3.json HTTP/...      1
13  IG01 POST /UOCP/person/ServiceTest3.json HTTP/...      1
14  IG01 POST /UOCP/person/ServiceTest3.json HTTP/...      1
15  IG01 POST /UOCP/person/ServiceTest3.json HTTP/...      1
16  IG01 POST /UOCP/person/ServiceTest5.json HTTP/...      1
17  IG01 POST /UOCP/person/ServiceTest9.json HTTP/...      1
18  IG02 POST /UOCP/base/ServiceTest11.json HTTP/1...      1
19  IG02 POST /UOCP/base/ServiceTest2.json HTTP/1....      1, Empty DataFrame
Columns: [dt, cmdb_id, message_display]
Index: [])```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
import numpy as np

# Reuse loaded dataframes if present; otherwise load from CSVs
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')
try:
    error_df
except NameError:
    error_df = pd.read_csv('error_logs.csv')

# Timezone and window
tz = pytz.timezone('Asia/Shanghai')
start = pd.Timestamp('2021-03-09 05:00:00').tz_localize(tz)
end = pd.Timestamp('2021-03-09 05:30:00').tz_localize(tz)

# Helper to compute p95, window, and earliest exceed per source
def compute_earliest_exceeds(df, cmdb_col, name_col, source_label):
    # Ensure types
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
    # Global p95 using full series (require grouping columns)
    p95_df = (
        df
        .groupby([cmdb_col, name_col], dropna=False)['value']
        .quantile(0.95)
        .reset_index()
        .rename(columns={'value': 'p95'})
    )
    # Datetime column localized to Asia/Shanghai
    df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    # Filter to window
    window_df = df[(df['dt'] >= start) & (df['dt'] <= end)].copy()
    # Merge p95 into window rows
    window_with_p95 = window_df.merge(p95_df, on=[cmdb_col, name_col], how='left')
    # Keep only rows that meet or exceed p95 (and p95 not null)
    exceed_rows = window_with_p95[window_with_p95['p95'].notna() & (window_with_p95['value'] >= window_with_p95['p95'])].copy()
    if exceed_rows.empty:
        # return empty DataFrame with expected columns and cached intermediates
        return (
            pd.DataFrame(columns=['source','cmdb_id','name','p95','earliest_exceed_ts','value_at_earliest_exceed']),
            p95_df, window_df, window_with_p95, exceed_rows
        )
    # Sort by dt ascending so first per group is earliest
    exceed_rows = exceed_rows.sort_values('dt')
    # For value_at_earliest_exceed pick the first row per (cmdb_id,name)
    first_exceeds = (
        exceed_rows
        .groupby([cmdb_col, name_col], dropna=False, as_index=False)
        .first()  # after sorting, .first gives earliest dt row
    )
    # Build result DataFrame with unified column names
    result = pd.DataFrame({
        'source': source_label,
        'cmdb_id': first_exceeds[cmdb_col],
        'name': first_exceeds[name_col],
        'p95': first_exceeds['p95'],
        'earliest_exceed_ts': first_exceeds['dt'],
        'value_at_earliest_exceed': first_exceeds['value']
    })
    return result, p95_df, window_df, window_with_p95, exceed_rows

# Compute for metric.csv
metric_exceed_df, metric_p95_df, metric_window, metric_window_with_p95, metric_exceed_rows = compute_earliest_exceeds(
    metric_df, 'cmdb_id', 'kpi_name', 'metric'
)

# Compute for trace.csv
trace_exceed_df, trace_p95_df, trace_window, trace_window_with_p95, trace_exceed_rows = compute_earliest_exceeds(
    trace_df, 'cmdb_id', 'trace_name', 'trace'
)

# Compute for log.csv
log_exceed_df, log_p95_df, log_window, log_window_with_p95, log_exceed_rows = compute_earliest_exceeds(
    log_df, 'cmdb_id', 'log_name', 'log'
)

# Combine the exceed dataframes (only pairs with an exceed within window are included)
combined_exceeds = pd.concat([metric_exceed_df, trace_exceed_df, log_exceed_df], ignore_index=True, sort=False)

# Prepare earliest error_log row in-window (if any)
# Ensure error_df has dt column and numeric timestamp
error_df['timestamp'] = pd.to_numeric(error_df['timestamp'], errors='coerce').astype('Int64')
error_df['dt'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
error_window_df = error_df[(error_df['dt'] >= start) & (error_df['dt'] <= end)].copy()

if not error_window_df.empty:
    earliest_error_row = error_window_df.sort_values('dt').iloc[0]
    err_row = pd.DataFrame([{
        'source': 'error_log',
        'cmdb_id': earliest_error_row['cmdb_id'],
        'name': np.nan,
        'p95': np.nan,
        'earliest_exceed_ts': earliest_error_row['dt'],
        'value_at_earliest_exceed': str(earliest_error_row['message'])[:200]
    }])
    combined_all = pd.concat([combined_exceeds, err_row], ignore_index=True, sort=False)
else:
    combined_all = combined_exceeds.copy()

# Sort combined table by earliest_exceed_ts ascending and limit to top 20
combined_all = combined_all.dropna(subset=['earliest_exceed_ts']).sort_values('earliest_exceed_ts', ascending=True)
combined_top20 = combined_all.loc[:, ['source','cmdb_id','name','p95','earliest_exceed_ts','value_at_earliest_exceed']].head(20)

# Cache full intermediate tables (variables already assigned): metric_p95_df, metric_window, metric_window_with_p95, metric_exceed_rows, trace_p95_df, trace_window, trace_window_with_p95, trace_exceed_rows, log_p95_df, log_window, log_window_with_p95, log_exceed_rows, error_window_df

# Display the compact combined table (top 20)
combined_top20

```
Out[6]:
```
summary = (
    "Combined earliest-exceed results (top 20) are dominated by metric entries with their first >=P95 hits at the start of the incident window (2021-03-09 05:00:00+08).\n\n"
    "Notable items:\n"
    "- IG01: JVM-Memory_7778_JVM_Memory_HeapMemoryMax hit P95 at 05:00:00 with value ~2.04e9 (true high memory value).\n"
    "- Many Mysql01 metrics (multiple Handler/Innodb counters) show earliest exceedances at 05:00:00; several of these have P95 = 0 and observed value = 0 at that time (this can be a baseline artifact — treat zero P95 results cautiously).\n"
    "- Redis01 filesystem metrics (FSAvail, FSUsedSpace, FSCapacity, inode metrics) also show earliest >=P95 at 05:00:00 with non-zero values (e.g., FSAvail ~3730, FSUsedSpace ~140), indicating noteworthy filesystem values at window start.\n\n"
    "Summary interpretation and guidance:\n"
    "- The earliest anomalous signals are concentrated at the very start of the window and are mostly from metric data (IG01, Mysql01, Redis01). IG01 JVM heap max and Redis filesystem metrics are actionable non-zero signals worth investigating.\n"
    "- Many Mysql01 entries showing P95=0 should be interpreted with caution because a zero baseline makes the >=P95 test uninformative; verify whether these are real anomalies or artifacts of metric normalization.\n"
    "- Next steps: inspect IG01 JVM memory state and Redis filesystem usage around 05:00, and correlate with traces/logs for Tomcat/Mysql to find causal relationships."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

source  cmdb_id                                               name           p95        earliest_exceed_ts value_at_earliest_exceed
0    metric     IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09 2021-03-09 05:00:00+08:00             2040135680.0
398  metric  Mysql01        Mysql-MySQL_3306_Handler Savepoint Rollback  0.000000e+00 2021-03-09 05:00:00+08:00                      0.0
399  metric  Mysql01                    Mysql-MySQL_3306_Handler Update  5.250000e-02 2021-03-09 05:00:00+08:00                      0.1
829  metric  Redis01  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSAvail...  3.730000e+03 2021-03-09 05:00:00+08:00                   3730.0
401  metric  Mysql01              Mysql-MySQL_3306_Innodb Row Lock Time  0.000000e+00 2021-03-09 05:00:00+08:00                      0.0
402  metric  Mysql01             Mysql-MySQL_3306_Innodb Row Lock Waits  0.000000e+00 2021-03-09 05:00:00+08:00                      0.0
828  metric  Redis01       OSLinux-OSLinux_FILESYSTEM_-boot_FSUsedSpace  1.400000e+02 2021-03-09 05:00:00+08:00                    140.0
397  metric  Mysql01                 Mysql-MySQL_3306_Handler Savepoint  0.000000e+00 2021-03-09 05:00:00+08:00                      0.0
404  metric  Mysql01  Mysql-MySQL_3306_Innodb buffer pool pages flushed  1.799700e-01 2021-03-09 05:00:00+08:00                   1.0667
406  metric  Mysql01     Mysql-MySQL_3306_Innodb buffer pool pages misc  0.000000e+00 2021-03-09 05:00:00+08:00                      0.0
407  metric  Mysql01    Mysql-MySQL_3306_Innodb buffer pool pages total  0.000000e+00 2021-03-09 05:00:00+08:00                      0.0
408  metric  Mysql01     Mysql-MySQL_3306_Innodb buffer pool read ahead  0.000000e+00 2021-03-09 05:00:00+08:00                      0.0
409  metric  Mysql01  Mysql-MySQL_3306_Innodb buffer pool read ahead...  0.000000e+00 2021-03-09 05:00:00+08:00                      0.0
410  metric  Mysql01  Mysql-MySQL_3306_Innodb buffer pool read ahead...  0.000000e+00 2021-03-09 05:00:00+08:00                      0.0
826  metric  Redis01        OSLinux-OSLinux_FILESYSTEM_-boot_FSCapacity  1.541700e+01 2021-03-09 05:00:00+08:00                   15.417
827  metric  Redis01  OSLinux-OSLinux_FILESYSTEM_-boot_FSInodeUsedPe...  4.715000e-01 2021-03-09 05:00:00+08:00                   0.4715
396  metric  Mysql01                  Mysql-MySQL_3306_Handler Rollback  0.000000e+00 2021-03-09 05:00:00+08:00                      0.0
831  metric  Redis01  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSInode...  4.200000e-03 2021-03-09 05:00:00+08:00                   0.0042
394  metric  Mysql01                  Mysql-MySQL_3306_Handler Read Rnd  0.000000e+00 2021-03-09 05:00:00+08:00                      0.0
379  metric  Mysql01                        Mysql-MySQL_3306_Com Update  5.250000e-02 2021-03-09 05:00:00+08:00                      0.1```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing DataFrames if present, otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')

# Timezone and window (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start = pd.Timestamp('2021-03-09 05:00:00').tz_localize(tz)
end = pd.Timestamp('2021-03-09 05:30:00').tz_localize(tz)

# Ensure dtypes and timezone-aware datetimes
for df in (trace_df, log_df):
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
    df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Compute or reuse global P95s for trace and log
try:
    trace_p95
except NameError:
    trace_p95 = (
        trace_df
        .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
        .quantile(0.95)
        .reset_index()
        .rename(columns={'value': 'p95'})
    )

try:
    log_p95
except NameError:
    log_p95 = (
        log_df
        .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
        .quantile(0.95)
        .reset_index()
        .rename(columns={'value': 'p95'})
    )

# Target series lists
trace_targets = [
    ('IG01', 'trace.to_Tomcat04.row_count'),
    ('IG01', 'trace.to_Tomcat01.row_count'),
    ('IG01', 'trace.self.row_count'),
    ('Tomcat04', 'trace.from_IG01.row_count'),
    ('Tomcat01', 'trace.from_IG01.row_count'),
    ('MG02', 'trace.to_dockerB2.row_count'),
    ('dockerB2', 'trace.from_MG02.row_count'),
    ('dockerB1', 'trace.self.row_count'),
]

log_targets = [
    ('Tomcat04', 'log.row_count'),
    ('Tomcat02', 'log.row_count'),
    ('Tomcat02', 'log.error_count'),
]

# Helper to gather up to first N exceed rows per series
def get_exceeds(df, p95_df, cmdb_col, name_col, targets, source_label, max_per_series=5):
    rows = []
    for cmdb_id, name in targets:
        # find global p95
        p95_row = p95_df[(p95_df[cmdb_col] == cmdb_id) & (p95_df[name_col] == name)]
        if p95_row.empty:
            continue
        p95_val = float(p95_row['p95'].iloc[0])
        if np.isnan(p95_val):
            continue
        # filter df to series and window
        subset = df[(df['cmdb_id'] == cmdb_id) & (df[name_col] == name) & (df['dt'] >= start) & (df['dt'] <= end)].copy()
        if subset.empty:
            continue
        # find rows where value >= p95
        exceed = subset[subset['value'] >= p95_val].sort_values('dt').head(max_per_series)
        if exceed.empty:
            continue
        for _, r in exceed.iterrows():
            rows.append({
                'source': source_label,
                'cmdb_id': cmdb_id,
                'name': name,
                'global_p95': p95_val,
                'earliest_exceed_ts': r['dt'],
                'value_at_each_exceed': r['value']
            })
    if not rows:
        return pd.DataFrame(columns=['source','cmdb_id','name','global_p95','earliest_exceed_ts','value_at_each_exceed'])
    return pd.DataFrame(rows)

# Get exceeds for trace and log targets
trace_exceeds = get_exceeds(trace_df, trace_p95, 'cmdb_id', 'trace_name', trace_targets, 'trace', max_per_series=5)
log_exceeds = get_exceeds(log_df, log_p95, 'cmdb_id', 'log_name', log_targets, 'log', max_per_series=5)

# Combine results
combined = pd.concat([trace_exceeds, log_exceeds], ignore_index=True, sort=False)

# Sort by earliest_exceed_ts ascending and limit to top 20
combined_sorted = combined.dropna(subset=['earliest_exceed_ts']).sort_values('earliest_exceed_ts', ascending=True)
result_top = combined_sorted.loc[:, ['source','cmdb_id','name','global_p95','earliest_exceed_ts','value_at_each_exceed']].head(20)

# Cache full intermediate tables (already stored in variables): trace_p95, log_p95, trace_exceeds, log_exceeds, combined_sorted

# Display compact result
result_top

```
Out[7]:
```
Summary of earliest exceed events (global P95 computed from full files; window 2021-03-09 05:00–05:30 +08). For each target series I list the global P95 and the earliest exceed rows (value >= P95), up to 5 rows per series:

Trace series
- IG01 / trace.to_Tomcat04.row_count — global_p95 = 4.0  
  earliest exceeds: 05:02:00 (4.0), 05:03:00 (4.0), 05:07:00 (5.0)

- IG01 / trace.to_Tomcat01.row_count — global_p95 = 6.0  
  earliest exceeds: 05:06:00 (6.0), 05:19:00 (6.0)

- IG01 / trace.self.row_count — global_p95 = 31.9  
  earliest exceeds: 05:06:00 (34.0)

- Tomcat04 / trace.from_IG01.row_count — global_p95 = 4.0  
  earliest exceeds: 05:02:00 (4.0), 05:03:00 (4.0), 05:07:00 (5.0)

- Tomcat01 / trace.from_IG01.row_count — global_p95 = 6.0  
  earliest exceeds: 05:06:00 (6.0), 05:19:00 (6.0)

- MG02 / trace.to_dockerB2.row_count — global_p95 = 3.0  
  earliest exceeds: 05:02:00 (3.0), 05:06:00 (3.0)

- dockerB2 / trace.from_MG02.row_count — global_p95 = 3.0  
  earliest exceeds: 05:02:00 (3.0), 05:06:00 (3.0)

- dockerB1 / trace.self.row_count — global_p95 = 136.0  
  earliest exceeds: 05:02:00 (136.0), 05:03:00 (185.0)

Log series
- Tomcat04 / log.row_count — global_p95 = 8.0  
  earliest exceeds: 05:02:00 (8.0), 05:03:00 (8.0), 05:07:00 (8.0)

- Tomcat02 / log.row_count — no exceed rows in-window (no rows met or exceeded global P95 in the window)

- Tomcat02 / log.error_count — no exceed rows in-window

Observations (compact)
- Many target trace and log series first exceeded their historical P95 shortly after 05:02 and around 05:06; some repeats occur later (e.g., 05:07, 05:19).  
- dockerB1 shows a large self.row_count spike (136 → 185) early in the window.  
- Tomcat04 log.row_count and several IG→Tomcat and MG→dockerB row_count traces repeatedly exceeded P95 during 05:02–05:07.  
- Tomcat02 log series had no detected exceed events within the window.

All full intermediate tables (per-series P95s and filtered/exceed rows) are cached in the kernel as requested; above is the compact table of results.

The original code execution output of IPython Kernel is also provided below for reference:

source   cmdb_id                         name  global_p95        earliest_exceed_ts  value_at_each_exceed
0   trace      IG01  trace.to_Tomcat04.row_count         4.0 2021-03-09 05:02:00+08:00                   4.0
22    log  Tomcat04                log.row_count         8.0 2021-03-09 05:02:00+08:00                   8.0
20  trace  dockerB1         trace.self.row_count       136.0 2021-03-09 05:02:00+08:00                 136.0
17  trace  dockerB2    trace.from_MG02.row_count         3.0 2021-03-09 05:02:00+08:00                   3.0
8   trace  Tomcat04    trace.from_IG01.row_count         4.0 2021-03-09 05:02:00+08:00                   4.0
14  trace      MG02  trace.to_dockerB2.row_count         3.0 2021-03-09 05:02:00+08:00                   3.0
1   trace      IG01  trace.to_Tomcat04.row_count         4.0 2021-03-09 05:03:00+08:00                   4.0
23    log  Tomcat04                log.row_count         8.0 2021-03-09 05:03:00+08:00                   8.0
21  trace  dockerB1         trace.self.row_count       136.0 2021-03-09 05:03:00+08:00                 185.0
9   trace  Tomcat04    trace.from_IG01.row_count         4.0 2021-03-09 05:03:00+08:00                   4.0
18  trace  dockerB2    trace.from_MG02.row_count         3.0 2021-03-09 05:06:00+08:00                   3.0
15  trace      MG02  trace.to_dockerB2.row_count         3.0 2021-03-09 05:06:00+08:00                   3.0
3   trace      IG01  trace.to_Tomcat01.row_count         6.0 2021-03-09 05:06:00+08:00                   6.0
11  trace  Tomcat01    trace.from_IG01.row_count         6.0 2021-03-09 05:06:00+08:00                   6.0
6   trace      IG01         trace.self.row_count        31.9 2021-03-09 05:06:00+08:00                  34.0
24    log  Tomcat04                log.row_count         8.0 2021-03-09 05:07:00+08:00                   8.0
10  trace  Tomcat04    trace.from_IG01.row_count         4.0 2021-03-09 05:07:00+08:00                   5.0
2   trace      IG01  trace.to_Tomcat04.row_count         4.0 2021-03-09 05:07:00+08:00                   5.0
12  trace  Tomcat01    trace.from_IG01.row_count         6.0 2021-03-09 05:19:00+08:00                   6.0
4   trace      IG01  trace.to_Tomcat01.row_count         6.0 2021-03-09 05:19:00+08:00                   6.0```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing metric_df if present; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# Ensure correct dtypes and timezone-aware datetime column
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Targets
targets = ['Tomcat04', 'IG01']

# 1) For each target cmdb_id list unique kpi_name values with their counts (limit top 50)
kpi_counts = (
    metric_df[metric_df['cmdb_id'].isin(targets)]
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values(['cmdb_id','count'], ascending=[True, False])
)
kpi_counts_top50 = kpi_counts.head(50)  # compact view

# 2A) Compute global P95 per (cmdb_id, kpi_name) using the full metric series (global thresholds)
p95_df = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value':'p95'})
)

# Restrict p95 to our targets (keep all kpi names for these two hosts)
p95_sub = p95_df[p95_df['cmdb_id'].isin(targets)].copy()

# 2B) Filter to incident window (inclusive)
start = pd.Timestamp('2021-03-09 05:00:00').tz_localize(tz)
end = pd.Timestamp('2021-03-09 05:30:00').tz_localize(tz)
metric_window = metric_df[(metric_df['dt'] >= start) & (metric_df['dt'] <= end)].copy()

# 2C) Merge p95 into windowed rows and compute exceed flag
metric_window_with_p95 = metric_window.merge(p95_sub, on=['cmdb_id','kpi_name'], how='right', suffixes=('','_p95_merge'))

# Note: After right-merge, rows that had no points in window will have NaN for timestamp/value/dt but will preserve p95.
# For proper per-row exceed computation, we only consider rows where 'value' is notna.
metric_window_with_p95['exceed'] = False
# For rows where value exists, set exceed = value >= p95
mask_val = metric_window_with_p95['value'].notna() & metric_window_with_p95['p95'].notna()
metric_window_with_p95.loc[mask_val, 'exceed'] = metric_window_with_p95.loc[mask_val, 'value'] >= metric_window_with_p95.loc[mask_val, 'p95']

# Aggregate per (cmdb_id, kpi_name) within window (including series with zero points)
def agg_group(g):
    total_pts = int(g['value'].count())  # count non-null values in window
    cnt_ex = int(g['exceed'].sum())
    if cnt_ex > 0:
        # earliest exceed timestamp among rows where exceed is True
        earliest = g.loc[g['exceed'], 'dt'].min()
    else:
        earliest = pd.NaT
    max_val = g['value'].max() if g['value'].count()>0 else np.nan
    return pd.Series({
        'total_points_in_window': total_pts,
        'count_exceed': cnt_ex,
        'earliest_exceed_ts': earliest,
        'max_value_in_window': max_val
    })

window_agg = (
    metric_window_with_p95
    .groupby(['cmdb_id','kpi_name','p95'], dropna=False)
    .apply(agg_group)
    .reset_index()
)

# Some kpi_names might not appear due to grouping nuances; ensure all p95_sub pairs are present by merging
full_agg = p95_sub.merge(window_agg, on=['cmdb_id','kpi_name','p95'], how='left')

# Fill missing window stats for pairs with no points in the window
full_agg['total_points_in_window'] = full_agg['total_points_in_window'].fillna(0).astype(int)
full_agg['count_exceed'] = full_agg['count_exceed'].fillna(0).astype(int)
# earliest_exceed_ts and max_value_in_window can remain NaT/NaN

# Sort results by earliest_exceed_ts ascending (NaT last) then count_exceed descending
result = full_agg.sort_values(by=['earliest_exceed_ts','count_exceed'], ascending=[True, False], na_position='last')

# Limit to top 50 rows for compact output and select requested columns
result_top50 = result.loc[:, ['cmdb_id','kpi_name','p95','total_points_in_window','count_exceed','earliest_exceed_ts','max_value_in_window']].head(50)

# Cache full intermediate tables in kernel:
# p95_df, p95_sub, metric_window, metric_window_with_p95, window_agg, full_agg

# Display compact outputs: kpi counts and result table (top 50)
kpi_counts_top50, result_top50

```
Out[8]:
```
```python
# Compact plain-English summary of the metric analysis focused on IG01 and Tomcat04
summary = (
    "Metric analysis (IG01 and Tomcat04) — concise findings:\n\n"
    "1) KPI coverage: The query listed up to 50 KPI series for IG01 and Tomcat04 (multiple JVM, OS, disk, network, process metrics).\n\n"
    "2) Strong signals on IG01:\n"
    "- JVM heap metrics: IG01 JVM-Memory_7778_JVM_Memory_HeapMemoryMax exceeded its global P95 at the very start of the window (2021-03-09 05:00:00+08) with value ≈2.04e9; JVM NoHeapMemoryUsed also exceeded P95 at 05:00 with a large value (~5.79e7).\n"
    "- Memory percent: OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc had p95 ≈97 and showed 22 in-window points exceeding P95 (earliest exceed at ~05:01), indicating sustained high memory usage.\n"
    "- Many IG01 series (CPU, process, network, and local-disk KPIs shown) report earliest exceedances at the start of the incident window (05:00–05:01), i.e., anomalies are present immediately at window start.\n\n"
    "3) Observations for Tomcat04:\n"
    "- Several Tomcat04 OS/network series show earliest >=P95 at ~05:01 (for example a TCP-CLOSE-WAIT metric with p95=0 had 19 points >=P95 starting 05:01). Note: p95=0 cases can be a baseline artifact and require cautious interpretation.\n"
    "- Tomcat04 also has other filesystem/network KPI entries with earliest exceed at window start (05:00–05:01) and some non-zero maxima recorded.\n\n"
    "4) Caveats and guidance:\n"
    "- Numerous series have global P95 equal to 0; in those cases 'count_exceed' may be high while the actual values are zero — treat such results as baseline artifacts rather than clear errors.\n"
    "- The clearest actionable hits are IG01 JVM memory and the high memory-percent metric (sustained near or above historic P95) and Tomcat04 non-zero disk/network maxima recorded at window start.\n"
    "- Next steps: investigate IG01 JVM heap/NoHeap memory and system memory pressure around 05:00–05:02, and check Tomcat04 socket/connection states and filesystem metrics around the same time to correlate with traces/logs."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                                           kpi_name  count
0     IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31
2     IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31
3     IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31
6     IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31
12    IG01                            OSLinux-CPU_CPU_CPULoad     31
..     ...                                                ...    ...
44    IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps     15
45    IG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...     15
46    IG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps     15
47    IG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead     15
50    IG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps     15

[50 rows x 3 columns],       cmdb_id                                           kpi_name           p95  total_points_in_window  count_exceed        earliest_exceed_ts  max_value_in_window
0        IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09                      31            31 2021-03-09 05:00:00+08:00         2.040136e+09
3        IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.786755e+07                      31            31 2021-03-09 05:00:00+08:00         5.786755e+07
66       IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00                      31            31 2021-03-09 05:00:00+08:00         0.000000e+00
77       IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  1.000000e+00                      31            31 2021-03-09 05:00:00+08:00         1.000000e+00
81       IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  3.000000e+00                      31            31 2021-03-09 05:00:00+08:00         3.000000e+00
..        ...                                                ...           ...                     ...           ...                       ...                  ...
51       IG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite  1.558667e+01                      15             1 2021-03-09 05:00:00+08:00         2.053333e+01
130  Tomcat04  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...  2.500000e-01                      11             1 2021-03-09 05:00:00+08:00         3.000000e-01
80       IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  3.000000e-01                      31            28 2021-03-09 05:01:00+08:00         3.000000e-01
63       IG01       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  9.700000e+01                      22            22 2021-03-09 05:01:00+08:00         9.700000e+01
151  Tomcat04     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00                      19            19 2021-03-09 05:01:00+08:00         0.000000e+00

[50 rows x 7 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[9]:

import pandas as pd
import re
import pytz
import numpy as np

# Reuse metric_df if present; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

tz = pytz.timezone('Asia/Shanghai')

# Ensure types and datetime column
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) List all Tomcat04 kpi_name values (unique) and their counts (top 50)
kpi_counts = (
    metric_df[metric_df['cmdb_id'] == 'Tomcat04']
    .groupby('kpi_name', dropna=False)['kpi_name']
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(50)
    .reset_index(drop=True)
)
kpi_counts_top50 = kpi_counts

# 2) Select Tomcat04 kpis whose name contains any of the specified substrings
substrs = ["Tomcat-Requests", "Tomcat-Threads", "Tomcat-"]
pattern = "|".join(re.escape(s) for s in substrs)
mask = kpi_counts_top50['kpi_name'].str.contains(pattern, regex=True)
selected_kpis = kpi_counts_top50.loc[mask, 'kpi_name'].unique().tolist()

# 3) Compute global P95 per (cmdb_id, kpi_name) using the full series (reuse if available)
try:
    p95_df
except NameError:
    p95_df = (
        metric_df
        .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
        .quantile(0.95)
        .reset_index()
        .rename(columns={'value': 'p95'})
    )

p95_sub = p95_df[(p95_df['cmdb_id'] == 'Tomcat04') & (p95_df['kpi_name'].isin(selected_kpis))].copy()

# 4) Filter to incident window (inclusive)
start = pd.Timestamp('2021-03-09 05:00:00').tz_localize(tz)
end = pd.Timestamp('2021-03-09 05:30:00').tz_localize(tz)
metric_window = metric_df[(metric_df['dt'] >= start) & (metric_df['dt'] <= end)].copy()

# 5) Merge p95 into windowed rows (right merge to keep all selected series even if no window points)
metric_window_with_p95 = metric_window.merge(p95_sub, on=['cmdb_id', 'kpi_name'], how='right', suffixes=('','_p95'))

# 6) Compute exceed flag (only where value exists)
metric_window_with_p95['exceed'] = False
mask_val = metric_window_with_p95['value'].notna() & metric_window_with_p95['p95'].notna()
metric_window_with_p95.loc[mask_val, 'exceed'] = metric_window_with_p95.loc[mask_val, 'value'] >= metric_window_with_p95.loc[mask_val, 'p95']

# 7) Aggregate per (cmdb_id, kpi_name)
def agg_grp(g):
    total_pts = int(g['value'].count())
    cnt_ex = int(g['exceed'].sum())
    earliest = g.loc[g['exceed'], 'dt'].min() if cnt_ex > 0 else pd.NaT
    max_val = g['value'].max() if total_pts > 0 else np.nan
    return pd.Series({
        'total_points_in_window': total_pts,
        'count_exceed': cnt_ex,
        'earliest_exceed_ts': earliest,
        'max_value_in_window': max_val
    })

window_agg = (
    metric_window_with_p95
    .groupby(['cmdb_id', 'kpi_name', 'p95'], dropna=False)
    .apply(agg_grp)
    .reset_index()
)

# 8) Ensure all selected kpis are present by merging p95_sub with window_agg
full_agg = p95_sub.merge(window_agg, on=['cmdb_id', 'kpi_name', 'p95'], how='left')

# Fill missing window stats for series with no points
full_agg['total_points_in_window'] = full_agg['total_points_in_window'].fillna(0).astype(int)
full_agg['count_exceed'] = full_agg['count_exceed'].fillna(0).astype(int)

# 9) Sort by earliest_exceed_ts ascending (NaT last) then count_exceed descending; limit top 20
result = full_agg.sort_values(by=['earliest_exceed_ts', 'count_exceed'], ascending=[True, False], na_position='last')
result_top20 = result.loc[:, ['cmdb_id','kpi_name','p95','total_points_in_window','count_exceed','earliest_exceed_ts','max_value_in_window']].head(20)

# Cache intermediates: p95_sub, metric_window, metric_window_with_p95, window_agg, full_agg

# Return compact outputs: Tomcat04 kpi counts (top 50) and the aggregated results for selected Tomcat kpis
kpi_counts_top50, result_top20

```
Out[9]:
```
Summary (Tomcat04 metric-focused, window 2021-03-09 05:00–05:30 +08):

- Inventory:
  - Tomcat04 has many KPI series (the query returned up to 50 unique kpi_name entries covering Tomcat-Sessions, Tomcat-Requests, Tomcat-Threads, OS, disk, CPU, etc.).

- Key findings for Tomcat-* series (global P95 computed from full file; window stats shown):
  - Tomcat-Requests_"http-nio-8003"_ErrorCount: global P95 = 640.0. All 31 points in the window are >= P95 (count_exceed = 31), earliest at 2021-03-09 05:00:00, max = 640.0. This indicates sustained error-counts at or above historic P95 throughout the window.
  - Tomcat-Requests_"http-nio-8003"_MaxTimeRequest (request max time): global P95 ≈ 368,554.0. All 31 window points >= P95, earliest 05:00:00, max ≈ 368,554 — meaning request latency was very high and sustained from the window start.
  - Tomcat-Threads_"http-nio-8003"_CurrentThreads: global P95 = 100.0, 31 points in-window, count_exceed = 31, earliest 05:00:00, max 100.0 — thread usage at or above historic P95 for the full window.
  - Tomcat-Threads_"http-nio-8003"_MaxThreads: global P95 = 500.0, 31 points in-window, count_exceed = 31, earliest 05:00:00, max 500.0 — indicates thread capacity configured at/high relative to baseline, observed at window start.
  - Some Tomcat-Requests series (ProcessingTime, RequestCount) show very large global P95 values and only a couple of exceed events late in the window (e.g., 05:29), meaning occasional extreme spikes at the window end rather than sustained behavior.
  - Many Tomcat-Sessions series returned global P95 = 0 with count_exceed = 31 (earliest 05:00). These zero-P95 cases are likely baseline artifacts — treat them cautiously (>=P95 with p95=0 does not necessarily indicate a meaningful anomaly).

- Interpretation / recommended next steps:
  - Tomcat04 shows sustained, actionable anomalies starting at the very start of the incident window: high request error counts and very large request max times, together with thread-level metrics at/above P95. This pattern suggests Tomcat04 experienced high latency and error conditions with elevated thread utilization beginning ~05:00.
  - Investigate Tomcat04 logs and traces around 05:00 for stack traces, long-running requests, thread pool exhaustion, and upstream load. Correlate with trace and log series (IG→Tomcat traces and Tomcat log.row_count/error_count) and with system metrics (CPU, memory, disk) to determine root cause.
  - Be cautious interpreting series with global P95 = 0; prioritize non-zero P95 signals (error_count, max request time, threads, request spikes).

If you want, I can list the compact top-20 rows of the Tomcat-* aggregate table (exact p95, counts, timestamps) in a simple table.

The original code execution output of IPython Kernel is also provided below for reference:

(                                             kpi_name  count
0   Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31
1   Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31
2   Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31
3   Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31
4   Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31
..                                                ...    ...
45                    OSLinux-CPU_CPU-1_SingleCpuidle     10
46                    OSLinux-CPU_CPU-1_SingleCpuUtil     10
47                    OSLinux-CPU_CPU-2_SingleCpuUtil     10
48  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...     10
49   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite     10

[50 rows x 2 columns],      cmdb_id                                           kpi_name          p95  total_points_in_window  count_exceed        earliest_exceed_ts  max_value_in_window
0   Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...        640.0                      31            31 2021-03-09 05:00:00+08:00                640.0
1   Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     368554.0                      31            31 2021-03-09 05:00:00+08:00             368554.0
4   Tomcat04    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter          0.0                      31            31 2021-03-09 05:00:00+08:00                  0.0
5   Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...          0.0                      31            31 2021-03-09 05:00:00+08:00                  0.0
6   Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...          0.0                      31            31 2021-03-09 05:00:00+08:00                  0.0
7   Tomcat04        Tomcat-Sessions_7441--_SESSIONActiveCounter          0.0                      31            31 2021-03-09 05:00:00+08:00                  0.0
8   Tomcat04     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter          0.0                      31            31 2021-03-09 05:00:00+08:00                  0.0
9   Tomcat04     Tomcat-Sessions_7441--_SESSIONRejectedSessions          0.0                      31            31 2021-03-09 05:00:00+08:00                  0.0
10  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...          0.0                      31            31 2021-03-09 05:00:00+08:00                  0.0
11  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...          0.0                      31            31 2021-03-09 05:00:00+08:00                  0.0
12  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...          0.0                      31            31 2021-03-09 05:00:00+08:00                  0.0
13  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...          0.0                      31            31 2021-03-09 05:00:00+08:00                  0.0
14  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...          0.0                      31            31 2021-03-09 05:00:00+08:00                  0.0
15  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...          0.0                      31            31 2021-03-09 05:00:00+08:00                  0.0
16  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...        100.0                      31            31 2021-03-09 05:00:00+08:00                100.0
17  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...          0.0                      31            31 2021-03-09 05:00:00+08:00                  0.0
18  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...        500.0                      31            31 2021-03-09 05:00:00+08:00                500.0
2   Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...  776976098.5                      31             2 2021-03-09 05:29:00+08:00          776976410.0
3   Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...    1151238.5                      31             2 2021-03-09 05:29:00+08:00            1151249.0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```